# Four-carrier question-conditioned budgets and OBF, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

Uses all of `data/plan4_dev.jsonl` (30 families, 90 variants, 60 unique A sheets).

A writes calculations once. Branches reuse that completed cache.

- `kv`: full completed cache (same written A as the compress run).
- `probe_32` / `probe_64`: question-conditioned headwise selection. The probe asks for each carrier's effective capacity, eligibility, and shipment conditions. It does **not** include B's prices or budget.
- `obf_32` / `obf_64`: the same probe scores and budgets, then residual injection.

Ranking scores prompt plus written tokens and discards newly appended probe keys. Do not use the old prompt-only probe helper here.

OBF is adapted to whole-cache selection. It is not a copy of the protected-reasoning (latent-tail) setup.

Dummy-token compress reference: kv_32 28/90, kv_64 29/90, full KV 82/90. This run judges question-conditioned compression.

If both small budgets stay poor, one predeclared budget-256 check is next. Do **not** start a fresh split from this notebook. Do not resume `results_stage_plan4_compress` or earlier plan4 folders.

Download `/kaggle/working/results_stage_plan4_probe`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_probe.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_probe.py").is_file() and (cand / "data" / "plan4_dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_probe.py and data/plan4_dev.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_probe.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("plan4_dev", sum(1 for _ in (REPO / "data" / "plan4_dev.jsonl").open(encoding="utf-8")))

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_probe")
print("Question-conditioned 32/64, then OBF on the same scores. No dummy-token ranking. No fresh split.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_probe", "--split", "plan4_dev"])
print("plan4 probe exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_probe"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_probe" / "run"
print("reading", src)
path = src / "summary.json"
if path.is_file():
    print(path.read_text(encoding="utf-8")[:16000])